# Chapter 3: Implicit Differentiation Through a Fit

## The Complete Pipeline

1. Draw samples from N(mu, sigma=1) where mu is trainable
2. Square the samples: y = x^2
3. Fit a Voigt PDF to the y-values using maximum likelihood → get gamma_fit
4. Loss = (gamma_fit - target)^2
5. Optimize mu via gradient descent

**The problem:** Step 3 (the fit) is an iterative optimization (L-BFGS) that blocks gradients.

**The solution:** Implicit differentiation — we differentiate *around* the fit using the optimality condition.

---
## The Pseudo-Voigt PDF

A Voigt profile is the convolution of a Gaussian and a Lorentzian. The **pseudo-Voigt** is a weighted sum:

$V(x) = \eta \cdot G(x) + (1-\eta) \cdot L(x)$

Fully differentiable in PyTorch.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

def pseudo_voigt_pdf(x, log_gamma, sigma_g, center, eta):
    """Pseudo-Voigt PDF. gamma = exp(log_gamma) to keep it positive."""
    gamma = torch.exp(log_gamma)
    gauss = torch.exp(-0.5 * ((x - center) / sigma_g) ** 2) / (sigma_g * torch.sqrt(torch.tensor(2 * torch.pi)))
    lorentz = (gamma / torch.pi) / ((x - center) ** 2 + gamma ** 2)
    return eta * gauss + (1 - eta) * lorentz

print('Ready')

---
## Step 1: Sample, Square, Fit

We draw samples from N(mu, 1), square them, and fit a Voigt PDF using maximum likelihood.

In [ ]:
n_samples = 100
mu_true = 5.0

# Generate data
x_samp = torch.randn(n_samples) + mu_true
y_data = x_samp ** 2

print(f'Samples: {n_samples}')
print(f'  y mean = {y_data.mean().item():.2f}, y std = {y_data.std().item():.2f}')

# Fit Voigt: log_gamma instead of gamma (keeps it positive naturally)
log_gamma = torch.tensor([2.0], requires_grad=True)  # gamma = exp(2) = 7.4
center_fit = torch.tensor([30.0], requires_grad=True)

opt = torch.optim.LBFGS([log_gamma, center_fit], max_iter=200, lr=0.5)

def nll_loss():
    """Negative log-likelihood of data under Voigt."""
    opt.zero_grad()
    pdf = pseudo_voigt_pdf(y_data.detach(), log_gamma, torch.tensor([5.0]),
                           center_fit, torch.tensor([0.5]))
    loss = -torch.sum(torch.log(pdf + 1e-10))
    loss.backward()
    return loss

print(f'Before fit: gamma = {torch.exp(log_gamma).item():.2f}')
opt.step(nll_loss)
print(f'After fit:  gamma = {torch.exp(log_gamma).item():.2f}, center = {center_fit.item():.2f}')

# Visualize
x_grid = torch.linspace(0, 80, 500)
pdf_vals = pseudo_voigt_pdf(x_grid, log_gamma.detach(), torch.tensor([5.0]),
                            center_fit.detach(), torch.tensor([0.5]))

plt.hist(y_data.numpy(), bins=15, density=True, alpha=0.5, label='Data')
plt.plot(x_grid.numpy(), pdf_vals.numpy(), '-', linewidth=2, 
         label=f'Voigt fit (gamma={torch.exp(log_gamma).item():.1f})')
plt.xlabel('y = x^2'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

---
## Step 2: Implicit Differentiation

At the optimal fit params $\theta^* = (\log\gamma^*, \text{center}^*)$, the inner loss gradient is zero:

$\frac{\partial L_{\text{inner}}}{\partial \theta} \big|_{\theta^*} = 0$

When we change `mu`, the data changes, and $\theta^*$ shifts. Differentiating the zero-gradient condition:

$\frac{\partial^2 L}{\partial \theta^2} \cdot \frac{d\theta^*}{d\mu} + \frac{\partial^2 L}{\partial \theta \partial \mu} = 0$

$\boxed{\frac{d\theta^*}{d\mu} = -\mathbf{H}^{-1} \cdot \frac{\partial^2 L}{\partial \theta \partial \mu}}$

In [ ]:
# ---- Step 2a: Generate data WITH grad tracking on mu ----
mu = torch.tensor([5.0], requires_grad=True)
torch.manual_seed(42)
x_samp = torch.randn(n_samples) + mu
y_data_grad = x_samp ** 2  # mu flows through to y_data

# ---- Step 2b: Fit (no grad tracking) ----
lg = torch.tensor([2.0], requires_grad=True)
cf = torch.tensor([30.0], requires_grad=True)
opt = torch.optim.LBFGS([lg, cf], max_iter=200, lr=0.5)

def closure():
    opt.zero_grad()
    pdf = pseudo_voigt_pdf(y_data_grad.detach(), lg, torch.tensor([5.0]),
                           cf, torch.tensor([0.5]))
    loss = -torch.sum(torch.log(pdf + 1e-10))
    loss.backward()
    return loss

opt.step(closure)
print(f'Fit: gamma = {torch.exp(lg).item():.4f}, center = {cf.item():.4f}')

# ---- Step 2c: Implicit differentiation ----
# Create fresh tensors at the optimum with grad tracking
lg_opt = lg.detach().clone().requires_grad_(True)
cf_opt = cf.detach().clone().requires_grad_(True)

# Inner loss: depends on lg_opt (requires_grad) AND y_data_grad (depends on mu)
pdf = pseudo_voigt_pdf(y_data_grad, lg_opt, torch.tensor([5.0]),
                       cf_opt, torch.tensor([0.5]))
loss_inner = -torch.sum(torch.log(pdf + 1e-10))

# Gradients at optimum
grad_lg, grad_cf = torch.autograd.grad(loss_inner, [lg_opt, cf_opt], create_graph=True)
print(f'Gradient at optimum: dL/dlog_gamma = {grad_lg.item():.4f}, dL/dcenter = {grad_cf.item():.4f}')

# Hessian matrix (2x2)
H_ll = torch.autograd.grad(grad_lg, lg_opt, retain_graph=True)[0]
H_lc = torch.autograd.grad(grad_lg, cf_opt, retain_graph=True)[0]
H_cl = torch.autograd.grad(grad_cf, lg_opt, retain_graph=True)[0]
H_cc = torch.autograd.grad(grad_cf, cf_opt, retain_graph=True)[0]

H = torch.tensor([[H_ll.item(), H_lc.item()], [H_cl.item(), H_cc.item()]])
print(f'\nHessian:\n{H}')

# Mixed derivatives d2L/(dtheta * dmu)
mixed_lg = torch.autograd.grad(grad_lg, mu, retain_graph=True)[0]
mixed_cf = torch.autograd.grad(grad_cf, mu, retain_graph=True)[0]
print(f'\nMixed derivatives:')
print(f'  d2L/(dlog_gamma * dmu) = {mixed_lg.item():.4f}')
print(f'  d2L/(dcenter * dmu) = {mixed_cf.item():.4f}')

# Solve: dtheta/dmu = -H^(-1) * mixed
mixed_vec = torch.tensor([[mixed_lg.item()], [mixed_cf.item()]])
dtheta_dmu = -torch.linalg.solve(H, mixed_vec)

print(f'\ndtheta*/dmu:')
print(f'  dlog_gamma/dmu = {dtheta_dmu[0,0].item():.4f}')
print(f'  dcenter/dmu = {dtheta_dmu[1,0].item():.4f}')

# Convert to dgamma/dmu (chain rule: gamma = exp(log_gamma))
dgamma_dmu = dtheta_dmu[0,0].item() * torch.exp(lg_opt).item()
print(f'  dgamma/dmu = {dgamma_dmu:.4f}')

---
## What We Learned

1. **Pipeline works:** N(mu,1) → square → Voigt fit → gamma_fit. Implicit differentiation gives us dgamma/dmu without backpropagating through L-BFGS.

2. **Two practical tricks:**
   - Use `log_gamma` instead of `gamma` so it stays positive naturally
   - Use enough samples (100+) for the fit to be stable

3. **Next step:** Add the outer loss L = (gamma_fit - 100)^2 and do full gradient descent on mu.